# ?? IMPORTANT: People-Only Filter Pipeline (Comparison Run)

This notebook executes the full **Leg 2 (w150 sparse)** sequence for the people-only filter experiment (Tasks 1-3).
It is intended to produce comparison data against the canonical baseline, **not** to replace it.

**Dependency Note:**
Consistent with historical canonical runs in this project, `torch`, `torchvision`, and `transformers` are **not pinned** here. Results will reflect whatever versions are currently standard in Google Colab's default environment. Explicitly pinning these ML dependencies has been identified as a deferred gap and is not fixed in this notebook.

# Extract and Train Sparse Embeddings (w150 Dataset)
This notebook performs the GPU-intensive extraction of DINOv3 and ReID embeddings for 4 sampled frames per shot (instead of a single keyframe) on the w150 dataset. It then trains the Graph Transformer to see if this sparse temporal representation improves the clustering/false-positive issue.

In [1]:
import os
import sys
from pathlib import Path
from google.colab import drive
import torch

# ==============================================================================
# Step 1: Mount Drive and set environment variable
# ==============================================================================
print("[STEP 1] Mounting Google Drive...")
drive.mount('/content/drive')
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/CCTV-Multiview-Project"
DRIVE_ROOT = Path(os.environ["DRIVE_ROOT"])


Mounted at /content/drive


In [2]:
# ==============================================================================
# Step 2: Verify GPU
# ==============================================================================
print("\n[STEP 2] Verifying GPU...")
if not torch.cuda.is_available():
    print("[FAIL] GPU is not available! Please change the runtime type to T4/A100 GPU and restart.")
    sys.exit(1)
print(f"[PASS] GPU detected: {torch.cuda.get_device_name(0)}")



[STEP 2] Verifying GPU...
[PASS] GPU detected: Tesla T4


In [3]:
# ==============================================================================
# Step 3: Setup Environment
# ==============================================================================
repo_dir = "/content/cctv-multiview-summarization"
if not os.path.exists(repo_dir):
    print(f"[INFO] Cloning repository to {repo_dir}...")
    !git clone https://github.com/Gautam-Shah306/cctv-multiview-summarization.git {repo_dir}

os.chdir(repo_dir)
!git fetch origin
!git checkout feature/stage1-object-detection
!git pull origin feature/stage1-object-detection
!pip install -q -r requirements-colab.txt

print("\n[INFO] Installing PyTorch Geometric...")
!pip install -q torch-geometric


[INFO] Cloning repository to /content/cctv-multiview-summarization...
Cloning into '/content/cctv-multiview-summarization'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 350 (delta 80), reused 135 (delta 38), pack-reused 168 (from 1)
Receiving objects: 100% (350/350), 73.14 MiB | 23.34 MiB/s, done.
Resolving deltas: 100% (131/131), done.
branch 'feature/stage1-object-detection' set up to track 'origin/feature/stage1-object-detection'.
Switched to a new branch 'feature/stage1-object-detection'
From https://github.com/Gautam-Shah306/cctv-multiview-summarization
 * branch            feature/stage1-object-detection -> FETCH_HEAD
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.7 MB/s eta 0:00:00

# ==============================================================================
# Step 3.5: Run Upstream People-Only Filters (w150)
# ==============================================================================
# These CPU steps are executed here to ensure an atomic, end-to-end Leg 2 run 
# inside a single Colab session, incorporating the new Task 1-2 filters.

In [4]:
print("\n[STEP 3.5] Running Upstream Filtering and Pair Generation...")
!python -m src.keyframe_selection --output-suffix w150 --shot-window-sec 6.0
!python -m src.generate_training_pairs --suffix w150


[STEP 3.5] Running Upstream Filtering and Pair Generation...
Novelty #1: Adaptive Keyframe Selection (DINO ViT + OSNet ReID)
Target View      : all
Shot Window      : 6.0 s (~150 frames)
Fusion Alpha     : 0.50 (DINO: 0.50, ReID: 0.50)
Adaptive Factor k: 1.00 (epsilon = mu - 1.00 * sigma)
Baseline Epsilon : 0.85
ReID Matching    : greedy (set-based)
ReID Aggregation : mean
Output Suffix    : w150
---------------------------------------------------------------------------
[INFO] Processing keyframe selection for view1...
[INFO] [view1] Excluded 438 candidate frames due to zero ReID detections (people-only filter).
       -> Decisions logged to: /content/cctv-multiview-summarization/data_manifests/keyframe_decisions_view1_w150.csv
[INFO] Processing keyframe selection for view2...
[INFO] [view2] Excluded 236 candidate frames due to zero ReID detections (people-only filter).
       -> Decisions logged to: /content/cctv-multiview-summarization/data_manifests/keyframe_decisions_view2_w150.c

In [5]:
# ==============================================================================
# Step 4: Extract Sparse-Sampled Embeddings
# ==============================================================================
print("\n[STEP 4] Extracting Sparse-Sampled Embeddings...")
!python -m src.extract_sparse_embeddings



[STEP 4] Extracting Sparse-Sampled Embeddings...
[INFO] Using device: cuda
[INFO] Identified 79 unique shots to process.
[INFO] Loading DINO model (facebook/dinov2-small)...
config.json: 100% 547/547 [00:00<00:00, 2.38MB/s]

model.safetensors: downloading bytes:  77% 68.4M/88.2M [00:00<00:00, 123MB/s, 4.20MB/s  ] 
model.safetensors: downloading bytes: 100% 83.7M/83.7M [00:01<00:00, 79.5MB/s, 8.00MB/s  ]
model.safetensors: reconstructing file: 100% 88.2M/88.2M [00:01<00:00, 83.8MB/s, 8.59MB/s  ]
Loading weights: 100% 223/223 [00:00<00:00, 6213.66it/s]
[INFO] Loading OSNet ReID model (osnet_x1_0)...
[INFO] Loaded ReID weights from /content/drive/MyDrive/CCTV-Multiview-Project/checkpoints/osnet_x1_0_market1501.pth (565/565 keys matched)
[INFO] Loading all detections from CSVs...
[INFO] Processing 104 target frames for view1...
[LOG] ReID Fallback Triggered: target frame 150 in view1 used detections from nearest frame 164 (distance: 14)
[LOG] ReID Fallback Triggered: target frame 299 in v

In [6]:
# ==============================================================================
# Step 5: Run 5-Fold CV Training (Sparse w150 dataset)
# ==============================================================================
print("\n[STEP 5] Running Graph Transformer 5-Fold Training on Sparse Data...")
!python -m src.train_graph_transformer_w150_sparse



[STEP 5] Running Graph Transformer 5-Fold Training on Sparse Data...
[INFO] Device: cuda.
[INFO] Building PyTorch Geometric graph data object with PCA...
[INFO] Dataset has 598 pairs (79 nodes, 133 dims). Starting 5-Fold CV...
\n--- FOLD 1/5 ---
Fold 1 Best Epoch: 89 | Val Loss: 0.6898 | F1: 0.5426 | Acc: 0.5083
Fold 1 CM: TP=70, TN=52, FP=68, FN=50
\n--- FOLD 2/5 ---
Fold 2 Best Epoch: 21 | Val Loss: 0.6932 | F1: 0.0000 | Acc: 0.5000
Fold 2 CM: TP=0, TN=120, FP=0, FN=120
\n--- FOLD 3/5 ---
Fold 3 Best Epoch: 293 | Val Loss: 0.6299 | F1: 0.6426 | Acc: 0.5875
Fold 3 CM: TP=89, TN=52, FP=68, FN=31
\n--- FOLD 4/5 ---
Fold 4 Best Epoch: 262 | Val Loss: 0.6591 | F1: 0.5837 | Acc: 0.5924
Fold 4 CM: TP=68, TN=73, FP=45, FN=52
\n--- FOLD 5/5 ---
Fold 5 Best Epoch: 7 | Val Loss: 0.6931 | F1: 0.0000 | Acc: 0.5042
Fold 5 CM: TP=0, TN=120, FP=0, FN=118
\n=======================================================
5-FOLD CROSS-VALIDATION RESULTS
Accuracy  : 0.5385 +/- 0.0421
Precision : 0.3352 +/- 0.2

In [7]:
print("\n[STEP 6] Persisting Models & Sparse Embeddings to Google Drive...")
import shutil
from pathlib import Path

# Copy models (Models inherently have new fold outputs, we can keep them or add _people_only)
# Let's add _people_only to model names to prevent overwrite
for fold in range(1, 6):
    local_model = Path(f"models/graph_transformer_w150_sparse_fold{fold}.pt")
    drive_model = DRIVE_ROOT / "models" / f"graph_transformer_w150_sparse_fold{fold}_people_only.pt"
    drive_model.parent.mkdir(parents=True, exist_ok=True)

    if local_model.exists():
        shutil.copy2(local_model, drive_model)
        size_mb = drive_model.stat().st_size / (1024 * 1024)
        print(f"[PASS] Fold {fold} Model successfully copied to Drive: {drive_model} ({size_mb:.2f} MB)")

# Copy newly extracted embeddings back to Drive (append _people_only)
dino_sparse = Path("data_manifests/training_features_dino_w150_sparse.npz")
reid_sparse = Path("data_manifests/training_features_reid_w150_sparse.npz")

for f in [dino_sparse, reid_sparse]:
    if f.exists():
        new_name = f.stem + "_people_only" + f.suffix
        drive_target = DRIVE_ROOT / "data_manifests" / new_name
        drive_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, drive_target)
        size_mb = drive_target.stat().st_size / (1024 * 1024)
        print(f"[PASS] Sparse embeddings successfully copied to Drive: {drive_target.name} ({size_mb:.2f} MB)")



[STEP 6] Persisting Models & Sparse Embeddings to Google Drive...
[PASS] Fold 1 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold1_people_only.pt (0.12 MB)
[PASS] Fold 2 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold2_people_only.pt (0.12 MB)
[PASS] Fold 3 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold3_people_only.pt (0.12 MB)
[PASS] Fold 4 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold4_people_only.pt (0.12 MB)
[PASS] Fold 5 Model successfully copied to Drive: /content/drive/MyDrive/CCTV-Multiview-Project/models/graph_transformer_w150_sparse_fold5_people_only.pt (0.12 MB)
[PASS] Sparse embeddings successfully copied to Drive: training_features_dino_w150_sparse_people_only.npz (0.14 M

In [11]:
import sys
from unittest.mock import patch
from src import run_graph_transformer_inference

print("\n[STEP 7] Running Final Inference (Video generation mocked for speed)...")
# We mock generate_video because rendering 6 videos is slow and unnecessary for a pure numerical comparison run.
# We also clear sys.argv so argparse doesn't crash on Jupyter connection arguments (e.g. -f kernel.json)
with patch('src.run_graph_transformer_inference.generate_video', lambda csv, out: print(f"[MOCK] Skipped generating video for {csv}")), patch.object(sys, 'argv', ['run_graph_transformer_inference.py']):
    run_graph_transformer_inference.main()



[STEP 7] Running Final Inference (Video generation mocked for speed)...


usage: colab_kernel_launcher.py [-h]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-d49c1dec-6b71-431b-a2a2-a2295d2f7cc9.json


SystemExit: 2

In [ ]:
print("\n[STEP 8] Copying ALL generated CSVs to Drive safely...")
import glob
import shutil
from pathlib import Path

# We grab all manifests (inference outputs + upstream outputs)
artifacts = glob.glob("data_manifests/*.csv") + glob.glob("data_manifests/*.mp4")
for artifact in artifacts:
    local_path = Path(artifact)
    
    # We want to sync inference outputs AND the new keyframe/training pairs
    if "graph_transformer" in artifact or "ruleBased" in artifact or "w150" in artifact:
        # Inject _people_only into the filename to prevent overwriting canonicals on Drive
        new_name = local_path.stem + "_people_only" + local_path.suffix
        drive_path = DRIVE_ROOT / "data_manifests" / new_name
        drive_path.parent.mkdir(parents=True, exist_ok=True)
        
        shutil.copy2(local_path, drive_path)
        print(f"[PASS] Copied {local_path.name} -> {new_name} on Drive.")
